# Transfer learning and CNN mechanics

Convolutional networks combine local pattern detectors with progressively larger receptive fields. Transfer learning reuses a representation learned on one dataset and adapts only part of it to a new task. These are related ideas, but they answer different questions: convolution describes the architecture; transfer learning describes how we initialize and optimize it.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision.models import resnet18
sns.set_theme(style='whitegrid',context='notebook'); torch.manual_seed(4)

## 1. One kernel, one output location

For a single-channel input, a 3×3 kernel at location `(i,j)` computes

`y[i,j] = Σ_u Σ_v K[u,v] x[i+u,j+v] + b`.

Padding decides how borders are treated. Stride decides how many input locations are skipped. The output size for one spatial axis is

`floor((N + 2P − K) / S) + 1`.

The code below uses the same image and kernel while changing one geometric choice at a time.

In [ ]:
image=torch.zeros(1,1,9,9); image[0,0,2:7,4]=1; image[0,0,4,2:7]=1
kernel=torch.tensor([[-1.,-1.,-1.],[0.,0.,0.],[1.,1.,1.]]).view(1,1,3,3)
settings=[('valid, stride 1',0,1),('same, stride 1',1,1),('same, stride 2',1,2)]
fig,axes=plt.subplots(1,4,figsize=(13,3))
axes[0].imshow(image[0,0],cmap='gray'); axes[0].set_title('input 9×9')
for ax,(title,padding,stride) in zip(axes[1:],settings):
    response=nn.functional.conv2d(image,kernel,padding=padding,stride=stride)[0,0]
    ax.imshow(response,cmap='coolwarm'); ax.set_title(f'{title}\n{tuple(response.shape)}')
for ax in axes: ax.axis('off')
plt.tight_layout()

## 2. Receptive fields grow through layers

A unit in the first 3×3 convolution sees 3 input pixels along each axis. If a second 3×3 convolution follows with stride 1, a unit in the second layer sees 5×5 input pixels: the first layer contributes a one-pixel ring on every side. With stride 2, the jump between neighboring receptive-field centers also changes.

For kernel `k`, stride `s`, and current jump `j`, the recurrence is:

`new_jump = j·s`, `new_field = old_field + (k−1)·j`.

This is why pooling and strided convolutions enlarge context while reducing spatial resolution.

In [ ]:
def receptive_field(layers):
    field=1; jump=1
    rows=[]
    for kernel,stride in layers:
        field += (kernel-1)*jump; jump *= stride; rows.append((kernel,stride,field,jump))
    return pd.DataFrame(rows,columns=['kernel','stride','receptive_field','center_jump'])
try:
    import pandas as pd
    print(receptive_field([(3,1),(2,2),(3,1),(2,2)]))
except NameError: pass

## 3. A residual block

A residual block learns a correction `F(x)` and returns `x + F(x)`. The skip path preserves an identity route for both activations and gradients. If the dimensions change, a projection on the skip path makes addition possible.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.body=nn.Sequential(nn.Conv2d(in_channels,out_channels,3,stride,padding=1,bias=False),nn.BatchNorm2d(out_channels),nn.ReLU(),nn.Conv2d(out_channels,out_channels,3,padding=1,bias=False),nn.BatchNorm2d(out_channels))
        self.skip=nn.Identity() if in_channels==out_channels and stride==1 else nn.Conv2d(in_channels,out_channels,1,stride,bias=False)
    def forward(self,x): return torch.relu(self.body(x)+self.skip(x))

x=torch.randn(2,3,16,16,requires_grad=True); y=ResidualBlock(3,8,2)(x); y.mean().backward()
print('shape:',tuple(y.shape),'input gradient norm:',round(x.grad.norm().item(),4))

## 4. Transfer learning as an optimization schedule

Replacing the final classifier creates a new task-specific head. A common schedule is:

1. freeze the backbone and train the new head;
2. unfreeze the later backbone blocks;
3. use a smaller learning rate for pretrained layers than for the new head.

The point is not that early features are permanently correct. It is that a small target dataset should not immediately destroy a representation that already captures edges, textures, and shapes.

In [ ]:
backbone=resnet18(weights=None)  # offline construction; use cached pretrained weights when available
for p in backbone.parameters(): p.requires_grad=False
features=backbone.fc.in_features; backbone.fc=nn.Linear(features,2)
print('head parameters:',sum(p.numel() for p in backbone.fc.parameters()))
for p in backbone.layer4.parameters(): p.requires_grad=True
optimizer=torch.optim.Adam([{'params':backbone.layer4.parameters(),'lr':1e-4},{'params':backbone.fc.parameters(),'lr':1e-3}])
print('learning rates:',[g['lr'] for g in optimizer.param_groups])

### Limits

The transfer-learning cell demonstrates parameter selection, not evidence that ResNet-18 is best for this task. A real comparison needs a dataset, preprocessing matched to the pretrained weights, frozen-versus-fine-tuned runs, and validation error analysis.